# EVE-NG three-router automation

This notebook keeps reusable output while calling the same safe CLI as `lab_automation.py`. Configure credentials in the environment before live actions; never place passwords in notebook cells.

In [ ]:
from notebook_runner import run_lab

print("Notebook automation helper loaded.")

## 1. Management SSH precheck

In [ ]:
from notebook_runner import run_lab

run_lab("precheck")

## 2. Review generated configurations

In [ ]:
from notebook_runner import run_lab

for scenario in ("baseline", "ospf", "ebgp", "bgp-only", "ospf-ebgp", "ibgp-rr"):
    print(f"\n===== {scenario.upper()} =====")
    run_lab("plan", "--scenario", scenario)

## 3. Back up and collect health

Live cells load `LAB_USERNAME`, `LAB_PASSWORD`, and optionally `LAB_SECRET` automatically from the git-ignored `.env` file.

In [ ]:
from notebook_runner import run_lab

run_lab("backup")
run_lab("health")

## 4. Apply a scenario

Set `APPLY_CHANGES = True` only after reviewing the plan. The cell prints a clear message when changes are disabled.

In [ ]:
from notebook_runner import run_lab

scenario = "ospf"
APPLY_CHANGES = False

if APPLY_CHANGES:
    run_lab("apply", "--scenario", scenario, "--apply")
else:
    print("No configuration sent: set APPLY_CHANGES = True after reviewing the plan.")

## 5. Add eBGP without touching OSPF

`bgp-only` sends only `router bgp` commands. It does not send interface or `router ospf` configuration. R1 uses AS65001, R2 AS65002, and R3 AS65003.

In [ ]:
from notebook_runner import run_lab

run_lab("plan", "--scenario", "bgp-only")

In [ ]:
from notebook_runner import run_lab

APPLY_BGP = False
if APPLY_BGP:
    run_lab("apply", "--scenario", "bgp-only", "--apply", "--save")
else:
    print("BGP apply skipped: set APPLY_BGP = True after reviewing the plan.")

In [ ]:
from notebook_runner import run_lab

# Displays OSPF neighbors plus BGP summary, table, routes, and loopback pings.
run_lab("verify", "--scenario", "bgp-only")

## 6. Verify OSPF adjacencies, routes, and loopback pings

In [ ]:
from notebook_runner import run_lab

scenario = "ospf"
run_lab("verify", "--scenario", scenario)

## 7. Cleanup routing protocols

Set `CLEANUP_ROUTING = True` only when removal is intended. Interface and management configuration is preserved.

In [ ]:
from notebook_runner import run_lab

CLEANUP_ROUTING = False
if CLEANUP_ROUTING:
    run_lab("cleanup", "--apply")
else:
    print("Cleanup skipped: set CLEANUP_ROUTING = True to remove routing protocols.")